# demo_circuit

> Circuit model with Hebbian compass and goal weights, reproducing the shape of Figs 3-5 (Dan et al., Neuron 2024).

This notebook follows `specs/circuit/`. **How to run:** [`specs/circuit/RUN.md`](../specs/circuit/RUN.md). Normal tests (`nbdev-test`) run 2 model flies per set-up; the full version (`nbdev-test --flags slow`, or running the notebook by hand) uses 20 flies and checks acceptance criteria AC-8 to AC-10.

Figures and metrics are written under `results/demo_circuit/` (gitignored; layout in RUN.md).

## Setup

Table 1 circuit parameters (`N = 32`, `kappa = pi`, `alphaC = 0.01`, `alphaG = 0.001`) and the run lengths of spec decision Q12.

In [ ]:
import math
from math import pi
import numpy as np
import matplotlib.pyplot as plt
from explRL.params import Params
from explRL.circuit import grid, sinGoal, outputs, goalStats, goalStep
from explRL.circuit_train import heldValue, goalSafeDistance, readout
from explRL.circuit_runs import runManyCircuit, runLengths
from explRL.envs import ZoneArena
from explRL.train import save_demo_figure, save_demo_metrics

DN = 'demo_circuit'
params = Params()
N, tS = params.nCirc, params.tS
th = grid(N)
L = runLengths()
seed, nRuns = 0, 2
artifact_mode = 'quick'
env = ZoneArena(N=N, symmetric=True)

## Policy curves (Fig 3c-d)

Saccade direction and mean fixation length against compass minus goal heading, for a weak and a strong sinusoidal goal.

In [ ]:
rel = np.rad2deg((th - th[8] + pi) % (2 * pi) - pi)
order = np.argsort(rel)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
for g in (0.25, 0.5):
    out = [outputs(sinGoal(g, th[8], N), c, params) for c in range(N)]
    dS = np.array([o['dS'] for o in out])
    fix = params.aF / np.array([o['nuF'] for o in out])
    axes[0].plot(rel[order], dS[order], label=f'g = {g}')
    axes[1].plot(rel[order], fix[order], label=f'g = {g}')
for ax, title in zip(axes, ['CW saccade probability dS', 'mean fixation aF / nuF (s)']):
    ax.set_xlabel('compass - goal (deg)'); ax.set_title(title); ax.legend()
plt.tight_layout()
save_demo_figure('policy_curves', fig=fig, mode=None, demo_name=DN)
plt.show()

## Goal plasticity (Fig 3g-h)

One 5 s fixation (`ceil(5 / dtG)` goal updates) at compass heading 90 degrees after reward and after punishment, then 50 rewarded fixations in a row at that heading.

In [ ]:
c_fix, w0 = 8, sinGoal(0.3, th[0], N)
nStep = math.ceil(5 / params.dtG)

def fixation(w, R):
    for _ in range(nStep):
        w = goalStep(w, c_fix, R, params)
    return w

seq = [w0]
for _ in range(50):
    seq.append(fixation(seq[-1], 1))
stats = [goalStats(w) for w in seq]
deg = np.rad2deg(th)
fig, axes = plt.subplots(1, 3, figsize=(12, 2.5))
for w, label in [(w0, 'start'), (fixation(w0, 1), 'after reward'), (fixation(w0, -1), 'after punishment')]:
    axes[0].plot(deg, w, label=label)
axes[0].axvline(deg[c_fix], color='k', ls=':'); axes[0].set_xlabel('goal neuron (deg)'); axes[0].legend(fontsize=7)
axes[1].plot(np.rad2deg([(s['thetaG'] + pi) % (2 * pi) - pi for s in stats])); axes[1].set_ylabel('goal heading (deg)')
axes[2].plot([s['strength'] for s in stats]); axes[2].set_ylabel('goal strength')
for ax in axes[1:]:
    ax.set_xlabel('rewarded fixations')
plt.tight_layout()
save_demo_figure('goal_plasticity', fig=fig, mode=None, demo_name=DN)
plt.show()

## Simulated set-ups

`'goal'` (Fig 3i), `'compass_asym'` and `'compass_sym'` (Fig 4), and `'both'` (Fig 5); recipes in design-low-level section 8. The quick cell keeps normal tests fast; the slow cell reruns with 20 flies.

In [ ]:
runs = {s: runManyCircuit(s, params, nRuns, seed=seed) for s in ('goal', 'compass_asym', 'compass_sym', 'both')}

In [ ]:
#| slow
artifact_mode, nRuns = 'full', 20
runs = {s: runManyCircuit(s, params, nRuns, seed=seed) for s in ('goal', 'compass_asym', 'compass_sym', 'both')}

In [ ]:
def meanHeld(rs, values, start, T):
    "Mean over flies of a recorded value held between iterations, on a 1 s grid."
    times = np.arange(0.0, T, 1.0)
    return times, np.mean([heldValue(r['history'], values(r), r[start], times, tS) for r in rs], 0)

dist = lambda r: goalSafeDistance(r['history'], env)
strength = lambda r: r['history'].strength

## Goal learning with a perfect compass (Fig 3i)

Distance from the goal (in arena coordinates) to the nearest safe-zone center, and goal strength, averaged over flies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
for ax, values, start, label in [(axes[0], dist, 'distStart', 'distance to safe (deg)'), (axes[1], strength, 'strStart', 'goal strength')]:
    ax.plot(*meanHeld(runs['goal'], values, start, L['goal'])); ax.set_xlabel('time (s)'); ax.set_ylabel(label)
plt.tight_layout()
save_demo_figure('goal_learning', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()

## Compass learning (Fig 4b-d)

Per scene: compass weights of fly 0, jump probability along the learned map (mean over flies), fixation readout of the frozen probe after training (mean over flies), and the probe jump rate before and after training. In the symmetric scene `pJ` at opposite headings adds up to 1, so stability shows in the probe jump rate, not in the map mean.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for row, (setup, title) in enumerate([('compass_asym', 'asymmetric'), ('compass_sym', 'symmetric')]):
    rs = runs[setup]
    axes[row, 0].imshow(rs[0]['wC'], aspect='auto', vmin=0, vmax=1, origin='lower')
    axes[row, 0].set_title(f'{title}: wC (ring x compass)')
    axes[row, 1].plot(np.rad2deg(th), np.mean([r['mapPJ'] for r in rs], 0))
    axes[row, 1].set_title('pJ along the map'); axes[row, 1].set_xlabel('arena heading (deg)'); axes[row, 1].set_ylim(0, 1)
    ro = [readout(r['probeAfter']['history'], N) for r in rs]
    axes[row, 2].plot(ro[0]['centers'], np.nanmean([o['fixDur'] for o in ro], 0), 'o-', ms=3)
    axes[row, 2].set_title('probe: mean fixation (s)'); axes[row, 2].set_xlabel('arena - goal (deg)')
    jr = [np.mean([r[k]['jumpRate'] for r in rs]) for k in ('probeBefore', 'probeAfter')]
    axes[row, 3].bar(['before', 'after'], jr); axes[row, 3].set_ylim(0, 0.6); axes[row, 3].set_title('probe jump rate')
plt.tight_layout()
save_demo_figure('compass_learning', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()

## Both systems together (Fig 5)

After asymmetric pre-training, every goal starts at a danger center (90 degrees from safe). Goal distance and strength over training (mean over flies), probe PI at 0, 300, 600, 900 and 1200 s, and the spread of single-fly PI scores at the end.

In [ ]:
rs = runs['both']
fig, axes = plt.subplots(1, 4, figsize=(15, 2.5))
axes[0].plot(*meanHeld(rs, dist, 'distStart', L['train'])); axes[0].set_ylabel('distance to safe (deg)')
axes[1].plot(*meanHeld(rs, strength, 'strStart', L['train'])); axes[1].set_ylabel('goal strength')
axes[2].plot(rs[0]['probeTimes'], np.mean([r['probePIs'] for r in rs], 0), 'o-'); axes[2].set_ylabel('probe PI')
axes[3].hist([r['probePIs'][-1] for r in rs], bins=8); axes[3].set_xlabel('final probe PI')
for ax in axes[:3]:
    ax.set_xlabel('training time (s)')
plt.tight_layout()
save_demo_figure('both', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()

## Measures

In [ ]:
m = lambda setup, key: float(np.mean([r[key] for r in runs[setup]]))
ac8 = dict(dist_start=m('goal', 'distStart'), dist_last_quarter=m('goal', 'distLastQ'),
           strength_start=m('goal', 'strStart'), strength_end=m('goal', 'strEnd'))
ac9 = dict(asym_map_pJ=m('compass_asym', 'mapPJmean'),
           asym_probe_jump_rate=float(np.mean([r['probeAfter']['jumpRate'] for r in runs['compass_asym']])),
           sym_probe_jump_rate=float(np.mean([r['probeAfter']['jumpRate'] for r in runs['compass_sym']])),
           sym_train_jump_rate=m('compass_sym', 'trainJumpRate'))
finalPI = [r['probePIs'][-1] for r in runs['both']]
ac10 = dict(dist_start=m('both', 'distStart'), dist_end=m('both', 'distEnd'),
            probe_PI_start=float(np.mean([r['probePIs'][0] for r in runs['both']])), probe_PI_end=float(np.mean(finalPI)),
            probe_PI_end_min=float(np.min(finalPI)), probe_PI_end_max=float(np.max(finalPI)),
            goal_minus_stablest_deg=float(np.mean(np.abs([r['goalMinusStablest'] for r in runs['both']]))))
for name, d in [('AC-8', ac8), ('AC-9', ac9), ('AC-10', ac10)]:
    print(name, {k: round(v, 3) for k, v in d.items()})
metrics = dict(mode=artifact_mode, seed=seed, nRuns=nRuns, lengths_s=L)
if artifact_mode == 'full':
    metrics.update({f'AC8_{k}': v for k, v in ac8.items()}, **{f'AC9_{k}': v for k, v in ac9.items()},
                   **{f'AC10_{k}': v for k, v in ac10.items()})
save_demo_metrics(metrics, mode=artifact_mode, demo_name=DN);

Acceptance checks (full run only):

In [ ]:
#| slow
assert ac8['dist_last_quarter'] < ac8['dist_start'] and ac8['strength_end'] > ac8['strength_start']   # AC-8
assert ac9['asym_map_pJ'] < 0.15 and ac9['asym_probe_jump_rate'] < 0.2                              # AC-9
assert ac9['sym_probe_jump_rate'] > 0.3
assert ac10['dist_end'] < ac10['dist_start']                                                           # AC-10
assert ac10['probe_PI_end'] > ac10['probe_PI_start'] and ac10['probe_PI_end'] > 0

## Summary

With Table 1 circuit parameters, the decided answers to spec Q1-Q12 and no tuning, 20 model flies per set-up (seeds 0-19) gave:

- **AC-8** (Fig 3i, perfect compass, 1200 s): the goal's distance to the nearest safe-zone center fell from 45.2 degrees at the start to 11.0 degrees (time average over the last quarter); goal strength rose from 0.39 to 0.82.
- **AC-9** (Fig 4): after 2400 s in the asymmetric scene, the mean jump probability along the learned map is 0.057 and a frozen probe jumps after 8.8% of saccades (50% with random weights). After 4800 s in the symmetric scene, the probe still jumps after 41.8% of saccades.
- **AC-10** (Fig 5): goals started at a danger center (90 degrees from safe) and ended 10.9 degrees from safe after 1200 s; the mean probe PI rose from -0.23 to 0.24. Single flies end between -0.20 and 0.64 (reported, not asserted). The goal ends on average 28 degrees from the most stable compass heading.

The full run takes about 21 s.